In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import PowerTransformer

warnings.filterwarnings('ignore')
# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


In [12]:
# import h5py
# 
# # 使用原始字符串避免路径转义问题
# file_path = r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\B_allResampledData.mat"
# # file_path = r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\B_betaResampled.mat"
# 
# # 打开 .mat 文件
# with h5py.File(file_path, "r") as f:
#     print("🔹 文件中所有顶层变量：")
#     print(list(f.keys()))
# 
#     print("\n🔹 文件层级结构：")
# 
# 
#     def print_tree(name, obj):
#         print(name)
# 
# 
#     f.visititems(print_tree)
# 
#     # 示例：如果文件中有变量 "data"，这样读取：
#     # data = f["data"][:]
#     # print("\n读取的 data：", data)


In [13]:
import h5py

file_path_A_beta = r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\A_betaResampled.mat"
file_path_B_beta = r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\B_betaResampled.mat"

with h5py.File(file_path_A_beta, "r") as f:
    dset = f["A_betaResampled"]
    # print(dset[0])
    # print("类型：", type(dset))
    # 2波段✘1000时间点✘3通道✘16subject✘7session（data结尾的是6个波段，beta的是high、 low两个）
    print("形状：", dset.shape)
    # print("数据类型：", dset.dtype)

with h5py.File(file_path_B_beta, "r") as f:
    dset = f["B_betaResampled"]
    # print(dset[0])
    # print("类型：", type(dset))
    # 2波段✘1000时间点✘3通道✘16subject✘7session（data结尾的是6个波段，beta的是high、 low两个）
    print("形状：", dset.shape)
    # print("数据类型：", dset.dtype)


形状： (2, 1000, 3, 16, 7)
形状： (2, 1000, 3, 16, 7)


In [14]:
import h5py

file_path_A_beta = r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\A_betaResampled.mat"
file_path_B_beta = r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\B_betaResampled.mat"

with h5py.File(file_path_A_beta, "r") as f:
    dset = f["A_betaResampled"]
    # 2波段✘1000时间点✘3通道(Fz、Cz、Pz)✘16subject✘7session（data结尾的是6个波段，beta的是high、 low两个）
    print("形状：", dset.shape)

with h5py.File(file_path_B_beta, "r") as f:
    dset = f["B_betaResampled"]
    # 2波段✘1000时间点✘3通道✘16subject✘7session（data结尾的是6个波段，beta的是high、 low两个）
    print("形状：", dset.shape)

形状： (2, 1000, 3, 16, 7)
形状： (2, 1000, 3, 16, 7)


In [15]:
names_A = ["曾奇", "王榕榕", "徐梓军", "徐文婷", "达格吉", "李天琦", "王威", "张渊博", "黄博文",
           "朱祖恩", "刘子皓", "李嘉文", "徐子焰", "李国荣", "杨珊珊", "付瑞晗"]

names_B = ["杜佳鑫", "冯科嘉", "陈妍", "胡浩男", "王子铭", "程腾宇", "朱娇娇", "祖丽米热", "毛华灏", "郭浚杰", "周鑫颜",
           "肖雨晨", "冯晓娅", "刘勇杰", "祝鹏程", "董嘉乐"]

In [16]:
def extract_eeg_features(filepath, names, group_label):
    with h5py.File(filepath, "r") as f:
        dset = f[list(f.keys())[0]]
        data = np.array(dset)  # shape: (2,1000,3,n_subject,7)

    bands = ["beta_low", "beta_high"]
    channels = ["Fz", "Cz", "Pz"]

    n_sub = data.shape[3]
    n_session = data.shape[4]

    rows = []
    for i in range(n_sub):
        subj_name = names[i]

        for bi, band in enumerate(bands):
            for ci, ch in enumerate(channels):
                for si in range(n_session):  # <<< 每个 session 单独处理
                    raw = data[bi, :, ci, i, si]  # shape (1000,)

                    feature_mean = raw.mean()
                    feature_std = raw.std()

                    rows.append({
                        "组别": group_label,
                        "受试者": subj_name,
                        "session": si + 1,  # session 从 1 到 7
                        "band": band,
                        "channel": ch,
                        "EEG_mean": feature_mean,
                        "EEG_std": feature_std,
                    })

    return pd.DataFrame(rows)


# ---------------- A 组 ----------------
df_A = extract_eeg_features(
    r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\A_betaResampled.mat",
    names_A,
    "Alcohol"
)

# ---------------- B 组 ----------------
df_B = extract_eeg_features(
    r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\B_betaResampled.mat",
    names_B,
    "Control"
)

df_eeg_long = pd.concat([df_A, df_B], ignore_index=True)
df_eeg_long

,组别,受试者,session,band,channel,EEG_mean,EEG_std
0,Alcohol,曾奇,1,beta_low,Fz,1.447357,0.425840
1,Alcohol,曾奇,2,beta_low,Fz,1.189647,0.340935
2,Alcohol,曾奇,3,beta_low,Fz,0.598739,0.287620
3,Alcohol,曾奇,4,beta_low,Fz,0.228910,0.493378
4,Alcohol,曾奇,5,beta_low,Fz,1.693787,0.131489
...,...,...,...,...,...,...,...
1339,Control,董嘉乐,3,beta_high,Pz,0.379189,0.280556
1340,Control,董嘉乐,4,beta_high,Pz,0.768079,0.538613
1341,Control,董嘉乐,5,beta_high,Pz,0.087621,0.061481
1342,Control,董嘉乐,6,beta_high,Pz,1.416907,1.423933


In [17]:
df_eeg = df_eeg_long.pivot_table(
    index=["组别", "受试者", "session"],
    columns=["band", "channel"],
    values="EEG_mean"
)

df_eeg.columns = [f"EEG_{b}_{c}_mean" for b, c in df_eeg.columns]
df_eeg = df_eeg.reset_index()


In [20]:
df_eeg.to_excel(
    r"E:\pycharm all files\眼动数据处理\相关性分析\数据\预处理后的数据\按session分开的_eeg.xlsx",
    index=False
)


In [31]:
df_other_subj = pd.read_excel(
    r"E:\pycharm all files\眼动数据处理\相关性分析\数据\预处理后的数据\eye&scl&ppg.xlsx"
)
df_other_subj.columns = ['组别', '受试者', 'session', 'SCL_mean_yj', 'AOI转换次数', '静态注释熵(SGE)',
                         '眼跳注视熵(GTE)', 'HR', 'SDNN', 'RMSSD', 'AMP']
df_other_subj

,组别,受试者,session,SCL_mean_yj,AOI转换次数,静态注释熵(SGE),眼跳注视熵(GTE),HR,SDNN,RMSSD,AMP
0,Alcohol,付瑞晗,1,0.008423,5.964286,0.793663,0.297438,17.819101,-3.260506,1.857342,-3100.994240
1,Alcohol,付瑞晗,2,0.008640,10.500000,0.884336,0.313038,5.119280,-15.197881,-21.082419,-2460.812152
2,Alcohol,付瑞晗,3,0.008858,15.035714,0.975010,0.328639,9.814463,11.623381,13.436624,-569.390018
3,Alcohol,付瑞晗,4,0.009075,19.571429,1.065683,0.344240,44.991362,25.830811,31.173141,-3871.876130
4,Alcohol,付瑞晗,5,0.009293,24.107143,1.156356,0.359840,10.378346,33.789581,41.231056,-985.276444
...,...,...,...,...,...,...,...,...,...,...,...
219,Control,陈妍,3,-0.070973,21.535714,1.521286,0.548103,-2.053871,38.025352,38.134196,-1057.457602
220,Control,陈妍,4,-0.088214,18.071429,1.326411,0.441623,-6.284982,-16.635726,-24.647037,-3145.236328
221,Control,陈妍,5,-0.105455,14.607143,1.131536,0.335144,-1.159217,17.134339,17.823007,-967.018514
222,Control,陈妍,6,-0.122697,11.142857,0.936661,0.228664,-5.534589,-22.075732,-34.098901,-6270.818380


In [32]:
df_final = df_other_subj.merge(
    df_eeg,
    on=["组别", "受试者", "session"],
    how="inner"
)

df_final.to_excel(
    r"E:\pycharm all files\眼动数据处理\相关性分析\数据\预处理后的数据\按session分开的_eeg&eye&scl&ppg.xlsx",
    index=False
)
df_final


,组别,受试者,session,SCL_mean_yj,AOI转换次数,静态注释熵(SGE),眼跳注视熵(GTE),HR,SDNN,RMSSD,AMP,EEG_beta_high_Cz_mean,EEG_beta_high_Fz_mean,EEG_beta_high_Pz_mean,EEG_beta_low_Cz_mean,EEG_beta_low_Fz_mean,EEG_beta_low_Pz_mean
0,Alcohol,付瑞晗,1,0.008423,5.964286,0.793663,0.297438,17.819101,-3.260506,1.857342,-3100.994240,-0.729108,-0.440792,1.180175,-0.683908,-0.657176,1.096477
1,Alcohol,付瑞晗,2,0.008640,10.500000,0.884336,0.313038,5.119280,-15.197881,-21.082419,-2460.812152,0.599527,1.112453,0.374152,0.599123,0.947222,0.145544
2,Alcohol,付瑞晗,3,0.008858,15.035714,0.975010,0.328639,9.814463,11.623381,13.436624,-569.390018,0.133105,1.244384,0.784893,-0.469845,1.081188,1.044413
3,Alcohol,付瑞晗,4,0.009075,19.571429,1.065683,0.344240,44.991362,25.830811,31.173141,-3871.876130,1.249114,1.324410,0.054951,0.609332,0.421649,-0.501940
4,Alcohol,付瑞晗,5,0.009293,24.107143,1.156356,0.359840,10.378346,33.789581,41.231056,-985.276444,1.462762,1.520320,0.673478,1.427771,1.266134,0.303502
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,Control,陈妍,3,-0.070973,21.535714,1.521286,0.548103,-2.053871,38.025352,38.134196,-1057.457602,0.480506,0.781104,0.844972,0.065712,0.332182,0.954549
220,Control,陈妍,4,-0.088214,18.071429,1.326411,0.441623,-6.284982,-16.635726,-24.647037,-3145.236328,0.858444,0.287205,0.201724,0.740466,0.307619,0.352925
221,Control,陈妍,5,-0.105455,14.607143,1.131536,0.335144,-1.159217,17.134339,17.823007,-967.018514,0.638032,0.071052,0.083395,0.153254,0.075354,0.068925
222,Control,陈妍,6,-0.122697,11.142857,0.936661,0.228664,-5.534589,-22.075732,-34.098901,-6270.818380,0.810403,0.183523,0.279126,0.459459,0.097723,0.332291
